If you're using DataSet_AIvsHHuman.csv, a typical machine learning workflow would be:

1. Define the problem

Assume the dataset contains examples labelled as either:

AI-generated → 1
Human-generated → 0

This becomes a binary classification problem where the model predicts whether a new sample was created by AI or a human.

2. Load and inspect the data

Check:

Number of rows
Feature columns
Target column
Missing values

3. Preprocess the data
If the dataset contains text
Python
from sklearn.feature_extraction.text import TfidfVectorizer
 
X = df["text"]
y = df["label"]
 
vectorizer = TfidfVectorizer(
max_features=5000,
stop_words='english'
)
 
X = vectorizer.fit_transform(X)
If the dataset contains numerical features
Python
X = df.drop("label", axis=1)
y = df["label"]
4. Split into training and test sets
Python
from sklearn.model_selection import train_test_split
 
X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.2,
random_state=42,
stratify=y
)

5. Train a classifier

A good starting model:

Python
from sklearn.ensemble import RandomForestClassifier
 
model = RandomForestClassifier(
n_estimators=200,
random_state=42
)
 
model.fit(X_train, y_train)

For text classification, these often perform even better:

Python
from sklearn.linear_model import LogisticRegression
 
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

or

Python
from sklearn.svm import LinearSVC
 
model = LinearSVC()
model.fit(X_train, y_train)

6. Evaluate the model
Python
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
 
predictions = model.predict(X_test)
 
print("Accuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

Useful metrics:

Accuracy
Precision
Recall
F1-score
ROC-AUC

7. Make predictions
Python
sample = [
"Artificial intelligence systems are transforming many industries."
]
 
sample_vector = vectorizer.transform(sample)
 
prediction = model.predict(sample_vector)
 
if prediction[0] == 1:
print("AI Generated")
else:
print("Human Written")

8. Improve performance

You could experiment with:

TF-IDF + Logistic Regression
Random Forest
XGBoost
LightGBM
BERT / RoBERTa transformers
Hyperparameter tuning with GridSearchCV

Example:

Python
from sklearn.model_selection import GridSearchCV
 
params = {
'C': [0.1, 1, 10]
}
 
grid = GridSearchCV(
LogisticRegression(max_iter=1000),
params,
cv=5
)
 
grid.fit(X_train, y_train)
 
print(grid.best_params_)
Recommended architecture

For an AI vs Human text dataset, the strongest progression is usually:

TF-IDF + Logistic Regression (baseline)
TF-IDF + SVM
Fine-tuned BERT/RoBERTa (best accuracy)

If you upload the actual CSV file, I can inspect its columns and build a complete training pipeline specifically for that dataset, including preprocessing, model selection, evaluation, and visualisations.

i want to be able to predict if writing conetent is ai or human generated, what steps should i take after splitting the data

Once you've split the data into training and test sets, the next steps are about turning the text into features, training models, evaluating them, and improving performance.

1. Convert text into numerical features

Machine learning models cannot understand raw text directly.

Start with TF-IDF
Python
from sklearn.feature_extraction.text import TfidfVectorizer
 
vectorizer = TfidfVectorizer(
max_features=10000,
ngram_range=(1,2),
stop_words='english'
)
 
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

This converts words and phrases into numerical vectors.

2. Establish a baseline model

Start simple before using deep learning.

Logistic Regression
Python
from sklearn.linear_model import LogisticRegression
 
model = LogisticRegression(max_iter=2000)
 
model.fit(X_train_tfidf, y_train)

Predict:

Python
y_pred = model.predict(X_test_tfidf)

Evaluate:

Python
from sklearn.metrics import classification_report
 
print(classification_report(y_test, y_pred))

Aim to record:

Accuracy
Precision
Recall
F1-score

F1-score is usually the most important metric for AI-vs-human detection.

3. Analyse mistakes

Build a confusion matrix:

Python
from sklearn.metrics import confusion_matrix
 
cm = confusion_matrix(y_test, y_pred)
print(cm)

Look at examples the model got wrong:

Python
misclassified = X_test[y_test != y_pred]

You'll often discover:

Human writing that sounds formal gets classified as AI.
AI writing with errors gets classified as human.

This analysis is extremely valuable.

4. Try stronger traditional models

Experiment with:

Linear SVM
Python
from sklearn.svm import LinearSVC
 
svm = LinearSVC()
 
svm.fit(X_train_tfidf, y_train)

For text classification, SVM often outperforms Random Forest.

XGBoost
Python
from xgboost import XGBClassifier

This can sometimes give a modest improvement.

5. Engineer AI-detection features

Many AI-generated texts exhibit patterns such as:

Sentence statistics
Python
avg_sentence_length
word_count
paragraph_count
Vocabulary richness
Python
unique_words / total_words
Readability scores
Python
Flesch Reading Ease
Gunning Fog
SMOG Index
Punctuation patterns

Count:

Python
!
?
;
:

AI often uses punctuation differently from humans.

6. Build a feature-combined model

Instead of only TF-IDF:

TF-IDF features
+
Readability features
+
Vocabulary richness
+
Sentence statistics


Combine them using:

Python
from scipy.sparse import hstack
 
X_combined = hstack([
X_train_tfidf,
handcrafted_features
])

This frequently performs better than TF-IDF alone.

7. Use cross-validation

Rather than trusting one train/test split:

Python
from sklearn.model_selection import cross_val_score
 
scores = cross_val_score(
model,
X_train_tfidf,
y_train,
cv=5,
scoring='f1'
)
 
print(scores.mean())

This gives a more reliable estimate.

8. Move to transformer models

Once the baseline works, fine-tune a transformer:

BERT
Python
bert-base-uncased
RoBERTa
Python
roberta-base
DeBERTa
Python
microsoft/deberta-v3-base

For AI-vs-human detection, DeBERTa is often among the strongest choices.

Training can be done with Hugging Face:

Python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
9. Evaluate using unseen sources

A common mistake is training and testing on text generated by the same AI models.

For example:

Train on GPT-3.5 + Human
Test on GPT-4 + Claude + Human

This checks whether the detector generalises.

Many AI detectors score 95%+ on familiar data but fall below 70% on unseen generators.

10. Deploy the model

Save everything:

Python
import joblib
 
joblib.dump(model, "ai_detector.pkl")
joblib.dump(vectorizer, "tfidf.pkl")

Prediction pipeline:

Python
text
↓
TF-IDF transform
↓
Model prediction
↓
Probability AI vs Human
Recommended progression

For a university or portfolio project:

TF-IDF + Logistic Regression
TF-IDF + Linear SVM
Add readability and linguistic features
Fine-tune DeBERTa
Compare all models using F1-score and ROC-AUC

This gives you a complete machine learning pipeline and demonstrates both traditional NLP and modern transformer approaches.

Absolutely. When people talk about an AI detector, they often jump straight to the machine learning model, but the statistical analysis is actually the foundation. In simple terms, you're trying to answer:

"What measurable differences exist between human writing and AI writing?"

Before training any model, you'd explore the data statistically.

Step 1: Look for patterns in the two groups

Imagine you separate your dataset into:

Human-written texts
AI-generated texts

Now measure characteristics of each text, such as:

Number of words
Average sentence length
Vocabulary diversity
Readability score
Number of commas, semicolons, etc.

Then compare the averages.

For example:

Feature	Human Average	AI AverageWords per sentence	18	24
Unique words (%)	65%	52%
Readability score	72	81

This immediately tells you whether there are visible differences.

Step 2: Measure variation, not just averages

A key statistical idea is that averages don't tell the whole story.

Suppose:

Humans average 20 words per sentence
AI averages 22 words per sentence

That's only useful if the groups don't overlap too much.

You would examine the standard deviation, which measures how spread out the values are.

Think of it like exam scores:

Class A average = 70%
Class B average = 72%

If most students score between 69% and 73%, the groups are basically identical.

The same applies to writing features.

You are looking for features where:

Human values cluster in one area
AI values cluster in another

Those features become useful predictors.

Step 3: Test whether differences are real

Sometimes differences happen by chance.

For example:

Human sentence length = 20
AI sentence length = 22

Is that a genuine difference or random noise?

This is where statistical tests help.

A common test is a t-test.

The output gives a p-value.

In simple language:

"How likely is it that this difference occurred randomly?"

A small p-value (typically below 0.05) suggests the difference is probably real.

For example:

Feature	p-valueSentence length	0.0001
Vocabulary diversity	0.002
Number of commas	0.68

Interpretation:

Sentence length matters
Vocabulary diversity matters
Commas probably don't matter
Step 4: Measure effect size

This is often overlooked but very important.

A feature can be statistically significant but practically useless.

Imagine:

Human average = 20.1
AI average = 20.2

With 100,000 documents that difference may produce a tiny p-value.

But the groups are still almost identical.

Effect size asks:

"How big is the difference?"

Large effect sizes are much more valuable than merely significant results.

Step 5: Correlation analysis

Next you'd look at how features relate to AI generation.

For example:

Feature	Correlation with AI labelSentence length	0.45
Readability	0.51
Vocabulary richness	-0.38

Positive correlation:

As feature increases, likelihood of AI writing increases.

Negative correlation:

As feature increases, likelihood of human writing increases.

The stronger the correlation, the more useful the feature may be.

Step 6: Understand probability

At its heart, an AI detector is making probability estimates.

Instead of saying:

"This is definitely AI."

The model really thinks:

"Based on the patterns I've learned, there is an 83% chance this resembles AI-generated writing."

Statistically, it's comparing the characteristics of a new text to patterns observed during training.

Step 7: Evaluate performance statistically

Once the model is trained, you'd calculate:

Accuracy

"What percentage of texts were classified correctly?"

Example:

900 correct out of 1000
Accuracy = 90%

Precision

"When the model says AI, how often is it correct?"

Example:

100 predictions = AI
90 actually AI

Precision = 90%

Recall

"How many AI texts did the model find?"

Example:

There are 100 AI texts
Model found 85

Recall = 85%

F1 Score

This combines precision and recall into a single metric.

For AI detection, F1 is often more useful than accuracy because datasets can be imbalanced.

Step 8: Feature importance

After training, you can ask:

"What statistical signals is the model using most?"

You might discover:

Vocabulary diversity
Readability score
Sentence length
Repetition frequency
Use of transition words

This becomes part of your analysis and discussion.

The key statistical story for your project

In layman's terms:

We first measured several characteristics of both human and AI-written texts. We then used statistical tests to determine which characteristics differed significantly between the two groups. Those meaningful features were used to train a machine learning model. The model learned the probability that a piece of writing belonged to either the AI or human category based on patterns observed in the training data. Finally, we evaluated how accurately the model could classify new, unseen pieces of text using metrics such as precision, recall, F1 score, and accuracy.

That's typically the level of statistical explanation lecturers and project supervisors are looking for because it demonstrates that you understand why the machine learning model works, not just how to run the code.